# 13. Trade Tape -- Unified SDR Enrichment

In [1]:
import nest_asyncio
nest_asyncio.apply()

import sys, os

# Add notebook dir (for _sdr_common) and project root (for SDRUtils)
_nb_dir = os.path.dirname(os.path.abspath("__file__"))
_project_root = os.path.normpath(os.path.join(_nb_dir, "..", ".."))
for p in [_nb_dir, _project_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import datetime
import pandas as pd
import _sdr_common as sdr

sdr.notebook_setup()

from SDRUtils.analytics.trade_tape import TradeTape

df = sdr.load_classified_trades(
    datetime.datetime(2026, 4, 1),
    datetime.datetime(2026, 4, 10),
)
tape = TradeTape(df)
enriched = tape.compute()
print(f"Raw: {len(df):,} trades | Enriched: {len(enriched.columns)} columns")
tape.summary()

  Using precomputed data from C:\Users\chris\clee\ARBS\notebooks\sdr\_precomputed_trades.parquet


Raw: 27,436 trades | Enriched: 85 columns


{'n_trades': 27436,
 'n_new_risk': 27436,
 'pct_new_risk': 100.0,
 'pct_compression': 0.0,
 'pct_ufro': np.float64(42.5),
 'pct_block': np.float64(2.8),
 'pct_capped': np.float64(2.0),
 'top_trade_types': {'OUTRIGHT': 8105,
  'CURVE': 7604,
  'SPREADOVER': 3715,
  'MATCHED_MATURITY': 3373,
  'FOMC': 1960},
 'venue_split': {'D2C': 23683, 'D2D': 3753},
 'ccp_split': {'LCH': 27436}}

In [2]:
for col in ["trade_type", "lifecycle_type", "venue", "ccp", "rate_index_clean"]:
    print(f"\n--- {col} ---")
    print(enriched[col].value_counts().to_string())


--- trade_type ---
trade_type
OUTRIGHT            8105
CURVE               7604
SPREADOVER          3715
MATCHED_MATURITY    3373
FOMC                1960
MAC                  895
IMM                  848
INVOICE_SWAP         528
FLY                  408

--- lifecycle_type ---
lifecycle_type
NEW_TRADE    27436

--- venue ---
venue
D2C    23683
D2D     3753

--- ccp ---
ccp
LCH    27436

--- rate_index_clean ---
rate_index_clean
SOFR         26975
FED_FUNDS      407
OTHER           54


In [3]:
clean = tape.clean_tape()
raw_dv01 = enriched["dv01"].sum()
clean_dv01 = clean["dv01"].sum()

print(f"Total DV01:  ${raw_dv01/1e9:.1f}B")
print(f"Clean DV01:  ${clean_dv01/1e9:.1f}B")
print(f"Noise:       ${(raw_dv01-clean_dv01)/1e9:.1f}B ({(1-clean_dv01/raw_dv01)*100:.0f}%)")
print(f"\nClean trades: {len(clean):,} / {len(enriched):,} ({len(clean)/len(enriched)*100:.0f}%)")

Total DV01:  $38238.0B
Clean DV01:  $11769.1B
Noise:       $26468.9B (69%)

Clean trades: 14,900 / 27,436 (54%)


In [4]:
from collections import Counter

all_flags = [f for flags in enriched["quality_flags"] for f in flags]
flag_counts = Counter(all_flags)
n = len(enriched)

print("Quality Flag Distribution:")
for flag, count in flag_counts.most_common():
    print(f"  {flag}: {count:,} ({count/n*100:.1f}%)")

print(f"\nUFRO: {enriched['is_ufro'].sum():,} ({enriched['is_ufro'].mean()*100:.1f}%)")
print(f"Block: {enriched['is_block'].sum():,} ({enriched['is_block'].mean()*100:.1f}%)")
print(f"Capped: {enriched['is_capped'].sum():,} ({enriched['is_capped'].mean()*100:.1f}%)")
print(f"Off-market: {enriched['is_off_market'].sum():,} ({enriched['is_off_market'].mean()*100:.1f}%)")

Quality Flag Distribution:
  UFRO: 11,674 (42.5%)
  OFF_MARKET: 7,385 (26.9%)
  CAPPED_NOTIONAL: 556 (2.0%)

UFRO: 11,674 (42.5%)
Block: 774 (2.8%)
Capped: 556 (2.0%)
Off-market: 7,385 (26.9%)


In [5]:
pkg = tape.package_summary()
print(f"Unique packages: {len(pkg):,}")
print(f"\nTop 15 package structures:")
print(pkg.groupby("package_structure")["total_dv01"].agg(["count","sum"]).sort_values("sum", ascending=False).head(15).to_string())

Unique packages: 4,669



Top 15 package structures:
                                                  count           sum
package_structure                                                    
2Y Matched_Maturity                                 285  1.574746e+12
4Y Matched_Maturity                                 378  1.130991e+12
7Y Matched_Maturity                                 460  1.080144e+12
3M Matched_Maturity                                 105  1.015553e+12
5Y Matched_Maturity                                 358  9.716575e+11
3Y Matched_Maturity                                 323  7.453244e+11
IMM_H2028 Matched_Maturity                          126  6.603698e+11
1M Matched_Maturity                                  35  3.957421e+11
7Y Invoice_Swap                                     188  3.837862e+11
1Y Matched_Maturity                                 110  3.651647e+11
10Y Matched_Maturity                                233  2.766658e+11
9Y Matched_Maturity                                  94  2.63

In [6]:
session_stats = enriched.groupby("execution_session").agg(
    trades=("dv01", "size"),
    total_dv01=("dv01", "sum"),
).sort_values("total_dv01", ascending=False)
session_stats["pct"] = (session_stats["total_dv01"] / session_stats["total_dv01"].sum() * 100).round(1)
print(session_stats.to_string())

                   trades    total_dv01   pct
execution_session                            
NY_PM                7981  1.443042e+13  37.7
NY_AM               11509  1.401539e+13  36.7
Late                 2184  5.351273e+12  14.0
London               3909  3.650990e+12   9.5
Asia                 1853  7.899354e+11   2.1


In [7]:
print(f"Total clusters: {enriched['cluster_id'].nunique():,}")
print(f"Multi-trade clusters: {(enriched['cluster_size'] > 1).sum():,} trades in clusters of 2+")

cluster_sizes = enriched.groupby("cluster_id").size()
print(f"\nCluster size distribution:")
print(cluster_sizes.value_counts().sort_index().head(10).to_string())

if enriched['is_multi_meeting_cluster'].any():
    n_multi = enriched['is_multi_meeting_cluster'].sum()
    print(f"\nMulti-FOMC-meeting clusters: {n_multi} trades")

Total clusters: 771
Multi-trade clusters: 27,242 trades in clusters of 2+

Cluster size distribution:
1     194
2     106
3      77
4      56
5      36
6      26
7      18
8      23
9      16
10     14

Multi-FOMC-meeting clusters: 22149 trades


In [8]:
cross = pd.crosstab(
    [enriched["trade_type"], enriched["rate_index_clean"]],
    enriched["venue"],
    values=enriched["dv01"],
    aggfunc="sum",
).fillna(0)
cross = cross / 1e6  # millions
print("DV01 Cross-tab (trade_type x rate_index x venue, $M):")
print(cross.round(0).to_string())

DV01 Cross-tab (trade_type x rate_index x venue, $M):
venue                                    D2C        D2D
trade_type       rate_index_clean                      
CURVE            FED_FUNDS          128576.0    10191.0
                 SOFR              4242332.0   926478.0
FLY              SOFR               266883.0     2048.0
FOMC             FED_FUNDS         2980174.0  4546776.0
                 SOFR              1981330.0   424853.0
IMM              SOFR               762555.0   195882.0
INVOICE_SWAP     SOFR               447953.0   596894.0
MAC              SOFR               411790.0      288.0
MATCHED_MATURITY FED_FUNDS          105445.0   139304.0
                 OTHER                   0.0        1.0
                 SOFR              8006801.0  1796414.0
OUTRIGHT         FED_FUNDS           95290.0   286629.0
                 OTHER               40710.0      110.0
                 SOFR              3178620.0  1565857.0
SPREADOVER       FED_FUNDS         1107948.0      

In [9]:
print("Top 20 enriched trade labels by DV01:")
label_dv01 = enriched.groupby("trade_label")["dv01"].sum().sort_values(ascending=False)
for label, dv01 in label_dv01.head(20).items():
    print(f"  ${dv01/1e6:>8.0f}M  {label}")

Top 20 enriched trade labels by DV01:
  $ 2128511M  FF IMM_M2026 FOMC_20260729
  $ 1701246M  FF UFRO IMM_M2026 FOMC_20260729
  $ 1648762M  SOFR UFRO spot 2Y
  $ 1197037M  SOFR UFRO spot 5Y
  $ 1098934M  FF UFRO spot FOMC_20260429
  $  847336M  SOFR UFRO spot 4Y
  $  688759M  SOFR spot 7Y
  $  685494M  SOFR UFRO spot 1Y
  $  683737M  SOFR UFRO spot 10Y
  $  652635M  SOFR spot 1Y
  $  592504M  SOFR spot 5Y
  $  589996M  SOFR UFRO spot 3Y
  $  522888M  SOFR spot 10Y
  $  512721M  FF 3W IMM_M2026
  $  433573M  SOFR spot 1M
  $  401089M  FF UFRO 3W IMM_M2026
  $  400449M  SOFR UFRO spot 7Y
  $  367980M  SOFR spot 2Y
  $  356324M  SOFR spot 3Y
  $  352466M  SOFR UFRO 3M IMM_H2028
